In [1]:


import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

BASE = Path('/kaggle/input/competitions/llm-agentic-legal-information-retrieval')

# ── LOAD DATA ─────────────────────────────────────────────────
print("Loading data...")
train_df = pd.read_csv(BASE / 'train.csv')
val_df   = pd.read_csv(BASE / 'val.csv')
test_df  = pd.read_csv(BASE / 'test.csv')

def parse_cit(v):
    if pd.isna(v) or str(v).strip() == '': return []
    return [x.strip() for x in str(v).split(';') if x.strip()]

train_df['cit_list'] = train_df['gold_citations'].apply(parse_cit)
val_df['cit_list']   = val_df['gold_citations'].apply(parse_cit)

# ── CORPUS ────────────────────────────────────────────────────
print("Building corpus...")
laws_df  = pd.read_csv(BASE / 'laws_de.csv', usecols=['citation'])
laws_set = set(laws_df['citation'].tolist())

all_court = set()
for chunk in pd.read_csv(BASE / 'court_considerations.csv',
                          chunksize=100_000, usecols=['citation']):
    all_court.update(chunk['citation'].tolist())

corpus_set = laws_set | all_court
print(f"Corpus: {len(corpus_set):,}")

# All abbreviations that actually appear in corpus
abbrev_set = set()
for cit in corpus_set:
    if cit.startswith('Art.'):
        abbrev_set.add(cit.strip().split()[-1])

# ── BLACKLIST ─────────────────────────────────────────────────
BLACKLIST = {c for c in corpus_set if 'KGTG' in c}
BLACKLIST |= set()  # add more if needed

global_freq = Counter()
for cits in train_df['cit_list']:
    global_freq.update(cits)
for cits in val_df['cit_list']:
    global_freq.update(cits)

global_score = {c: v for c, v in global_freq.items()
                if c not in BLACKLIST and c in corpus_set}

# ── FRENCH → GERMAN ABBREVIATION MAPPING ─────────────────────
FR_TO_DE = {
    # Major Swiss federal law abbreviations (FR → DE)
    'LAI':   'IVG',   # Loi fédérale sur l'assurance-invalidité
    'LPGA':  'ATSG',  # Loi fédérale sur la partie générale du droit des assurances sociales
    'LAA':   'UVG',   # Loi fédérale sur l'assurance-accidents
    'LAMal': 'KVG',   # Loi fédérale sur l'assurance-maladie
    'LAVS':  'AHVG',  # Loi fédérale sur l'assurance-vieillesse et survivants
    'LPP':   'BVG',   # Loi fédérale sur la prévoyance professionnelle
    'LACI':  'AVIG',  # Loi fédérale sur l'assurance-chômage
    'LTF':   'BGG',   # Loi sur le Tribunal fédéral
    'CPC':   'ZPO',   # Code de procédure civile
    'CPP':   'StPO',  # Code de procédure pénale
    'CP':    'StGB',  # Code pénal
    'CC':    'ZGB',   # Code civil
    'CO':    'OR',    # Code des obligations
    'LP':    'SchKG', # Loi fédérale sur la poursuite pour dettes et la faillite
    'LDIP':  'IPRG',  # Loi fédérale sur le droit international privé
    'LDA':   'URG',   # Loi sur le droit d'auteur
    'LCD':   'UWG',   # Loi fédérale contre la concurrence déloyale
    'LPM':   'MSchG', # Loi sur la protection des marques
    'LIFD':  'DBG',   # Loi fédérale sur l'impôt fédéral direct
    'LIVA':  'MWSTG', # Loi fédérale régissant la taxe sur la valeur ajoutée
    'LFINMA':'FINMAG', # Loi sur l'Autorité fédérale de surveillance des marchés financiers
    'LBA':   'BankG', # Loi sur les banques
    'LSA':   'VAG',   # Loi sur la surveillance des assurances
    'LRFP':  'PrHG',  # Loi fédérale sur la responsabilité du fait des produits
    'LASuivD': 'AIG', # immigration
    'LEI':   'AIG',   # Loi sur les étrangers et l'intégration
    'LAsi':  'AsylG', # Loi sur l'asile
    'PA':    'VwVG',  # Loi fédérale sur la procédure administrative
    'LFors': 'ZPO',   # (old conflicts of jurisdiction law, now ZPO)
    'LREC':  'BGG',
    'LCR':   'SVG',   # Loi fédérale sur la circulation routière
    'LAA':   'UVG',
    'OJ':    'BGG',
    'PCF':   'ZPO',
    'LDFR':  'BGBB',
    'LPN':   'NHG',
    'LAT':   'RPG',
    'LArm':  'ArG',
    'LTrWap':'WG',
}

# Build reverse: also handle Abs./al. etc.
# French art format: "art. 17 LAI" → normalize to German format

# ── CO-CITATION ───────────────────────────────────────────────
print("Building co-citation index...")
cit_cooccur = defaultdict(Counter)
for cit_list in list(train_df['cit_list']) + list(val_df['cit_list']):
    clean = [c for c in cit_list if c not in BLACKLIST]
    s = set(clean)
    for c in clean:
        for o in s:
            if o != c:
                cit_cooccur[c][o] += 1

def coexpand(seeds, top=25):
    scores = Counter()
    seed_set = set(seeds)
    for c in seeds:
        w = 1 + global_score.get(c, 0) * 0.05
        for o, cnt in cit_cooccur.get(c, {}).items():
            if o not in BLACKLIST and o in corpus_set:
                scores[o] += cnt * w
    for c in seed_set:
        scores.pop(c, None)
    return list(seed_set) + [c for c, _ in scores.most_common(top)
                              if c in corpus_set]

# ── ABBREVIATION INDEX ────────────────────────────────────────
print("Building abbreviation index...")
abbrev_to_cits = defaultdict(list)
for cit in corpus_set:
    if cit.startswith('Art.') and cit not in BLACKLIST:
        a = cit.strip().split()[-1]
        if 'KGTG' not in a:
            abbrev_to_cits[a].append(cit)
for abbrev in abbrev_to_cits:
    abbrev_to_cits[abbrev].sort(key=lambda c: global_score.get(c, 0), reverse=True)

# ── REGEX EXTRACTION (FIXED) ──────────────────────────────────
# Pattern 1: Art. N [Abs. M] [lit. x] ABBREV  (standard)
ART_FULL = re.compile(
    r'\b[Aa]rt(?:icle|ikel|\.)\s*\.?\s*([\d]+[a-z]*)'
    r'(?:\s+(?:Abs(?:atz|\.)?|al\.?|para\.?)\s*([\d]+[a-z]*))?'
    r'(?:\s+(?:lit\.?|let\.?)\s*[a-z])?'
    r'\s+([A-ZÄÖÜ][A-Za-z0-9äöüÄÖÜ]{1,15})\b'
)
# Pattern 2: Art. N [Abs. M] — bare, no trailing abbrev
ART_BARE = re.compile(
    r'\b[Aa]rt(?:icle|ikel|\.)\s*\.?\s*([\d]+[a-z]*)'
    r'(?:\s+(?:Abs(?:atz|\.)?|al\.?|para\.?)\s*([\d]+[a-z]*))?'
    r'(?:\s+(?:lit\.?|let\.?)\s*[a-z])?\b'
)
# BGE pattern
BGE_PAT = re.compile(r'\bBGE\s+(\d+\s+[IVX]+\s+\d+)\s+E\.\s*([\d\.a-z]+)')
# Case number
CASE_PAT = re.compile(r'\b([1-9][A-Z]{1,3}_\d+/\d{4})\s+E\.\s*([\d\.a-z]+)')
# Standalone abbreviations
ABBR_PAT = re.compile(
    r'\b(StPO|ZGB|OR|StGB|BV|ZPO|BGG|SchKG|IVG|ATSG|'
    r'UVG|KVG|AHVG|BVG|AVIG|SVG|USG|DBG|MWSTG|VVG|'
    r'URG|DSG|IPRG|RPG|NHG|GwG|FusG|PatG|MSchG|ArG|'
    r'FINMAG|BankG|HMG|BetmG|AsylG|AIG|VwVG|FIDLEG|'
    r'StBOG|JStG|JStPO|GBV|BGFA|PrHG|VAG|EIMP|RAG|'
    r'UWG|BEHG|KAG|FinfraG|FINIG|BEG|VStG|PrHG|'
    r'VZAE|EOG|FamZG|DesG|MSchV|AHVV|UVV|HVUV|'
    r'LAI|LPGA|LAA|LAMal|LAVS|LPP|LACI|LTF|CPC|CPP|'
    r'CP(?!\w)|CC(?!\w)|CO(?!\w)|LP(?!\w)|LDIP|LDA|LCD|LPM|'
    r'LIFD|LIVA|LFINMA|LBA|LSA|LRFP|LEI|LAsi|LCR|PA(?!\w))\b'
)

def normalize_abbrev(raw):
    """Convert French/English abbreviation to German."""
    return FR_TO_DE.get(raw, raw)

def try_canon(art, abs_, abbrev):
    """Try to find canonical citation in corpus."""
    abbrev = normalize_abbrev(abbrev)
    if 'KGTG' in abbrev:
        return None
    candidates = []
    if abs_:
        c1 = f'Art. {art} Abs. {abs_} {abbrev}'
        candidates.append(c1)
    c2 = f'Art. {art} {abbrev}'
    candidates.append(c2)
    for c in candidates:
        if c in corpus_set and c not in BLACKLIST:
            return c
    return None

def extract_regex_advanced(query):
    """
    Multi-pass regex extraction:
    1. Full match with abbreviation
    2. Bare Art. N refs paired with abbreviations from context
    3. BGE / case numbers
    """
    text = str(query)
    found = []
    found_set = set()

    def add(c):
        if c and c in corpus_set and c not in BLACKLIST and c not in found_set:
            found.append(c)
            found_set.add(c)

    # Pass 1: Full article refs WITH explicit abbreviation
    for m in ART_FULL.finditer(text):
        art = m.group(1); abs_ = m.group(2) or ''; abbrev = m.group(3)
        c = try_canon(art, abs_, abbrev)
        if c:
            add(c)

    # Pass 2: Bare article refs — pair with nearest abbreviation in context
    # Find all abbreviations and their positions first
    abbr_positions = [(m.start(), normalize_abbrev(m.group(1)))
                      for m in ABBR_PAT.finditer(text)
                      if normalize_abbrev(m.group(1)) in abbrev_set]

    for m in ART_BARE.finditer(text):
        art = m.group(1); abs_ = m.group(2) or ''
        pos = m.start()
        # Find closest abbreviation within 150 chars
        best_abbrev = None; best_dist = 999
        for apos, abbrev in abbr_positions:
            dist = abs(apos - pos)
            if dist < best_dist and dist < 150:
                best_dist = dist; best_abbrev = abbrev
        if best_abbrev:
            c = try_canon(art, abs_, best_abbrev)
            if c:
                add(c)

    # Pass 3: BGE refs
    for m in BGE_PAT.finditer(text):
        c = f'BGE {m.group(1).strip()} E. {m.group(2).strip()}'
        if c in corpus_set:
            add(c)

    # Pass 4: Case numbers
    for m in CASE_PAT.finditer(text):
        c = f'{m.group(1)} E. {m.group(2).strip()}'
        if c in corpus_set:
            add(c)

    return found

def extract_abbrevs(query):
    raw = set(ABBR_PAT.findall(str(query)))
    result = set()
    for r in raw:
        g = normalize_abbrev(r)
        if 'KGTG' not in g and g in abbrev_set:
            result.add(g)
    return list(result)

# ── DOMAIN KEYWORD MAP ────────────────────────────────────────
DOMAIN_KW = {
    # Inheritance / wills / estate
    'testament':    ['ZGB','IPRG'],
    'testator':     ['ZGB','IPRG'],
    'holograph':    ['ZGB'],
    'will ':        ['ZGB'],
    'heir':         ['ZGB','IPRG','SchKG'],
    'inherit':      ['ZGB','IPRG'],
    'estate':       ['ZGB','IPRG','SchKG'],
    'bequest':      ['ZGB'],
    'legatee':      ['ZGB'],
    'intestate':    ['ZGB'],
    'disinherit':   ['ZGB'],
    'forced share': ['ZGB'],
    'reserved':     ['ZGB'],
    # Family / child
    'custody':      ['ZGB','ZPO','IPRG'],
    'visitation':   ['ZGB','ZPO'],
    'divorce':      ['ZGB','ZPO','IPRG'],
    'matrimoni':    ['ZGB','ZPO','IPRG'],
    'maintenance':  ['ZGB','ZPO','OR'],
    'child support':['ZGB','ZPO'],
    'parental':     ['ZGB','ZPO'],
    'alimony':      ['ZGB','ZPO'],
    'marital':      ['ZGB','ZPO','IPRG'],
    'separation':   ['ZGB','ZPO'],
    'spouse':       ['ZGB','ZPO'],
    'overnight':    ['ZGB','ZPO'],
    # Disability / social insurance
    'invalidity':   ['IVG','ATSG','BGG'],
    'disability':   ['IVG','ATSG','BGG'],
    'invalide':     ['IVG','ATSG'],
    'rehab':        ['IVG','ATSG'],
    'earning capac':['IVG','ATSG'],
    'work capac':   ['IVG','ATSG','UVG'],
    'insured':      ['UVG','KVG','IVG','ATSG'],
    'accident':     ['UVG','ATSG','SVG'],
    'occupational': ['UVG','ATSG','ArG'],
    'pension':      ['BVG','AHVG','ATSG'],
    'social insur': ['ATSG','IVG','UVG','AHVG'],
    'asthma':       ['UVG','IVG','ATSG'],
    'allerg':       ['UVG','IVG','ATSG'],
    'respiratory':  ['UVG','IVG','ATSG'],
    'granulomatos': ['UVG','ATSG'],
    # Criminal
    'criminal':     ['StGB','StPO','BGG'],
    'pre-trial':    ['StPO','BGG'],
    'pretrial':     ['StPO','BGG'],
    'detention':    ['StPO','BGG'],
    'collusion':    ['StPO'],
    'flight risk':  ['StPO','BGG'],
    'accused':      ['StPO','StGB'],
    'assault':      ['StGB','StPO'],
    'theft':        ['StGB','StPO'],
    'robbery':      ['StGB','StPO'],
    'offence':      ['StGB','StPO'],
    'offense':      ['StGB','StPO'],
    'sentence':     ['StGB','StPO'],
    'conviction':   ['StGB','StPO'],
    'disloyal':     ['StGB','StPO'],
    'embezzl':      ['StGB','StPO'],
    'fraud':        ['StGB','StPO'],
    'sexual':       ['StGB','StPO'],
    'dna profile':  ['StPO'],
    'juvenile':     ['JStG','JStPO','StGB'],
    # Contract / civil
    'contract':     ['OR','ZGB','ZPO'],
    'lease':        ['OR','ZPO'],
    'tenancy':      ['OR','ZPO'],
    'landlord':     ['OR','ZPO'],
    'tenant':       ['OR','ZPO'],
    'employer':     ['OR','ArG'],
    'employee':     ['OR','ArG'],
    'employment':   ['OR','ArG'],
    'damages':      ['OR','ZPO'],
    'liability':    ['OR','PrHG','ZGB'],
    'mandate':      ['OR'],
    'work contract':['OR'],
    'defect':       ['OR','PrHG'],
    'product liab': ['PrHG','OR'],
    'bank':         ['BankG','OR','ZGB'],
    'forgery':      ['StGB','OR'],
    'forged':       ['StGB','OR'],
    'signature':    ['OR','ZGB'],
    # Bankruptcy / enforcement
    'bankrupt':     ['SchKG','OR'],
    'insolvenc':    ['SchKG','OR'],
    'creditor':     ['SchKG','OR'],
    'debt enforc':  ['SchKG'],
    'opposition':   ['SchKG','ZPO'],
    'lien':         ['ZGB','SchKG'],
    'mortgage':     ['ZGB','SchKG'],
    # Procedure / appeal
    'appeal':       ['BGG','ZPO','StPO'],
    'federal court':['BGG'],
    'supreme court':['BGG'],
    'jurisdiction': ['IPRG','ZPO','BGG'],
    'provisional':  ['ZPO','IPRG','ZGB'],
    'interim':      ['ZPO','IPRG'],
    'injunction':   ['ZPO','UWG'],
    'evidence':     ['ZPO','StPO'],
    # PIL
    'international':['IPRG','ZPO'],
    'foreign judgm':['IPRG','ZPO'],
    'recognition':  ['IPRG','ZPO'],
    'habitual resi':['IPRG','ZGB'],
    'choice of law':['IPRG'],
    'applicable law':['IPRG','ZPO'],
    # IP
    'trademark':    ['MSchG','UWG'],
    'copyright':    ['URG','UWG'],
    'domain name':  ['UWG','MSchG'],
    'unfair comp':  ['UWG'],
    # Tax
    'tax':          ['DBG','MWSTG','VStG'],
    'withholding':  ['VStG'],
    # Real property
    'land register':['ZGB','GBV'],
    'real estate':  ['ZGB','OR','GBV'],
    'building':     ['OR','ZGB','RPG'],
    'construct':    ['OR','ZGB','RPG'],
    'contractor':   ['OR','ZGB'],
    'statutory lien':['ZGB','OR'],
    # Asylum / immigration
    'asylum':       ['AsylG','AIG','VwVG'],
    'residence permit':['AIG','IPRG'],
    # AHV
    'ahv':          ['AHVG','ATSG'],
    'contribution': ['AHVG','ATSG'],
    'board member': ['AHVG','OR'],
    # Medical / doctor
    'medical':      ['UVG','IVG','ATSG','OR'],
    'physician':    ['UVG','IVG','ATSG','OR'],
    'treatment':    ['UVG','IVG','ATSG','OR'],
    'ophthalm':     ['OR'],
    'surgeon':      ['OR'],
    'standard of care':['OR'],
}

def get_domain_abbrevs(query):
    q = query.lower()
    found = set()
    for kw, abbrevs in DOMAIN_KW.items():
        if kw in q:
            found.update(abbrevs)
    return [a for a in found if a in abbrev_set and 'KGTG' not in a]

def abbrev_expand(direct_hits, abbrevs, top_per=20):
    results = []
    direct_set = set(direct_hits)
    for abbrev in abbrevs:
        cits = [c for c in abbrev_to_cits.get(abbrev, [])
                if c not in direct_set and c not in BLACKLIST]
        scored = []
        for cit in cits:
            co = sum(cit_cooccur.get(d, {}).get(cit, 0) for d in direct_hits)
            g  = global_score.get(cit, 0)
            scored.append((cit, co * 5 + g))
        scored.sort(key=lambda x: x[1], reverse=True)
        results.extend(c for c, _ in scored[:top_per])
    return results

# ── TF-IDF ────────────────────────────────────────────────────
print("Building TF-IDF...")
train_qs = train_df['query'].astype(str).tolist()
val_qs   = val_df['query'].astype(str).tolist()
test_qs  = test_df['query'].astype(str).tolist()
all_qs   = train_qs + val_qs + test_qs

tfidf_w = TfidfVectorizer(max_features=80_000, ngram_range=(1,2),
                           sublinear_tf=True, strip_accents='unicode', min_df=1)
tfidf_w.fit(all_qs)
train_wv = tfidf_w.transform(train_qs)
val_wv   = tfidf_w.transform(val_qs)
test_wv  = tfidf_w.transform(test_qs)

tfidf_c = TfidfVectorizer(max_features=60_000, ngram_range=(3,5),
                           sublinear_tf=True, analyzer='char_wb', min_df=2)
tfidf_c.fit(all_qs)
train_cv = tfidf_c.transform(train_qs)
val_cv   = tfidf_c.transform(val_qs)
test_cv  = tfidf_c.transform(test_qs)

def train_transfer(qvec_w, qvec_c, top_k=50, top_cit=80):
    sw = cosine_similarity(qvec_w, train_wv).flatten()
    sc = cosine_similarity(qvec_c, train_cv).flatten()
    s  = 0.5 * sw + 0.5 * sc
    idx = np.argsort(s)[::-1][:top_k]
    scores = Counter()
    for i in idx:
        sim = float(s[i])
        if sim < 0.02: continue
        for cit in train_df['cit_list'].iloc[i]:
            if cit not in BLACKLIST:
                scores[cit] += sim * sim
    return [(c, sc2) for c, sc2 in scores.most_common(top_cit)
            if c in corpus_set]

# ── SCORING ───────────────────────────────────────────────────
def score_query(query, qvec_w, qvec_c):
    scored = {}

    def add(items, weight):
        for item in items:
            c = item[0] if isinstance(item, tuple) else item
            s = float(item[1]) if isinstance(item, tuple) else 1.0
            if c not in corpus_set or c in BLACKLIST: continue
            scored[c] = scored.get(c, 0.0) + weight * s

    # 1. Direct regex (highest confidence)
    direct = extract_regex_advanced(query)
    add([(c, 1.0) for c in direct], weight=10.0)

    # 2. Explicit abbreviation expansion
    abbrevs = extract_abbrevs(query)
    if abbrevs:
        exp = abbrev_expand(direct, abbrevs, top_per=25)
        add([(c, 1.0) for c in exp], weight=3.0)

    # 3. Domain keyword expansion (always, but lower weight)
    domain_ab = get_domain_abbrevs(query)
    extra_ab   = list(set(domain_ab) - set(abbrevs))
    if extra_ab:
        dom_exp = abbrev_expand(direct + list(scored)[:5], extra_ab, top_per=15)
        add([(c, 1.0) for c in dom_exp], weight=1.5)

    # 4. Training query transfer
    t_hits = train_transfer(qvec_w, qvec_c, top_k=50, top_cit=80)
    add(t_hits, weight=1.5)

    # 5. Global freq micro-boost
    for c in list(scored):
        scored[c] += 0.03 * global_score.get(c, 0.0)

    # 6. Co-citation expansion on top seeds
    top15 = sorted(scored, key=scored.get, reverse=True)[:15]
    exp2  = coexpand(top15, top=30)
    for c in exp2:
        if c not in scored:
            scored[c] = 0.005 * global_score.get(c, 0.001)

    return sorted(scored, key=scored.get, reverse=True)

# ── F1 ────────────────────────────────────────────────────────
def f1_fn(pred, gt):
    p = set(pred); g = set(gt)
    if not p and not g: return 1.0
    if not p or  not g: return 0.0
    tp = len(p & g)
    pr = tp/len(p); rc = tp/len(g)
    return 2*pr*rc/(pr+rc) if (pr+rc) else 0.0

# ── VALIDATE ─────────────────────────────────────────────────
print(f"\nValidating {len(val_df)} queries...")
val_ranked = []
for i, (_, row) in enumerate(val_df.iterrows()):
    ranked = score_query(str(row['query']), val_wv[i], val_cv[i])
    val_ranked.append(ranked)

print("\nPer-query oracle:")
total_reach = total_gold = 0
for i, (_, row) in enumerate(val_df.iterrows()):
    gold    = row['cit_list']
    ranked  = val_ranked[i]
    reach   = set(ranked) & set(gold)
    best_f1 = 0.0; best_k = 1
    for k in range(1, min(len(ranked)+1, 300)):
        f = f1_fn(ranked[:k], gold)
        if f >= best_f1: best_f1 = f; best_k = k
    total_reach += len(reach); total_gold += len(gold)
    direct = extract_regex_advanced(str(row['query']))
    abbrevs = extract_abbrevs(str(row['query']))
    print(f"  val[{i}] oracle={best_f1:.3f} k={best_k} "
          f"reach={len(reach)}/{len(gold)} "
          f"regex={len(direct)} abbrevs={abbrevs[:4]}")
    if set(gold) - set(ranked):
        miss = sorted(set(gold) - set(ranked))
        print(f"    unreachable: {miss[:6]}")

print(f"\nOverall reachable: {total_reach}/{total_gold} "
      f"({100*total_reach/total_gold:.1f}%)")

# ── TUNE k ───────────────────────────────────────────────────
print("\nTuning k:")
best_k, best_f1 = 5, 0.0
for k in [5,8,10,12,15,20,25,30,35,40,45,50,60,70,80,100]:
    f1s = [f1_fn(val_ranked[i][:k], val_df.iloc[i]['cit_list'])
           for i in range(len(val_df))]
    mf1 = float(np.mean(f1s))
    print(f"  k={k:3d} → F1={mf1:.4f}")
    if mf1 > best_f1: best_f1 = mf1; best_k = k

# Adaptive k
def adaptive_k(query, base=best_k):
    d = len(extract_regex_advanced(str(query)))
    a = len(extract_abbrevs(str(query)))
    if d >= 5 or a >= 3: return min(base + 20, 100)
    if d == 0 and a == 0: return max(base, 20)
    return base

adap_f1s = [f1_fn(val_ranked[i][:adaptive_k(str(val_df.iloc[i]['query']))],
                   val_df.iloc[i]['cit_list'])
             for i in range(len(val_df))]
adap_f1  = float(np.mean(adap_f1s))
use_adap = adap_f1 > best_f1
final_f1 = adap_f1 if use_adap else best_f1
print(f"\n  Fixed k={best_k} → {best_f1:.4f}")
print(f"  Adaptive  → {adap_f1:.4f}")
print(f"  Using: {'adaptive' if use_adap else f'fixed-{best_k}'}")

print(f"\nDetailed debug:")
for i, (_, row) in enumerate(val_df.iterrows()):
    k    = adaptive_k(str(row['query'])) if use_adap else best_k
    pred = val_ranked[i][:k]
    gold = row['cit_list']
    hits = set(pred) & set(gold)
    direct = extract_regex_advanced(str(row['query']))
    print(f"\n  val[{i}] F1={f1_fn(pred,gold):.3f} hits={len(hits)}/{len(gold)} "
          f"direct={direct[:5]}")
    print(f"    missed: {sorted(set(gold)-set(pred))[:5]}")

# ── PREDICT TEST ─────────────────────────────────────────────
print(f"\nPredicting test...")
rows = []
for i, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
    ranked = score_query(str(row['query']), test_wv[i], test_cv[i])
    k      = adaptive_k(str(row['query'])) if use_adap else best_k
    pred   = [c for c in ranked[:k] if c not in BLACKLIST]
    if not pred:
        pred = [c for c,_ in Counter(global_score).most_common(k)
                if c not in BLACKLIST][:k]
    rows.append({'query_id': row['query_id'],
                 'predicted_citations': ';'.join(pred)})

sub = pd.DataFrame(rows)
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f"\n{'='*60}")
print(f"  Val F1 = {final_f1:.4f}  |  Rows = {len(sub)}")
print(f"{'='*60}")
print(sub.to_string())

Loading data...
Building corpus...
Corpus: 2,161,111
Building co-citation index...
Building abbreviation index...
Building TF-IDF...

Validating 10 queries...

Per-query oracle:
  val[0] oracle=0.467 k=18 reach=40/42 regex=1 abbrevs=['StPO']
    unreachable: ['BGE 137 IV 122 E. 4.2', 'BGE 143 IV 168 E. 5.1']
  val[1] oracle=0.426 k=11 reach=36/36 regex=0 abbrevs=['IVG']
  val[2] oracle=0.356 k=26 reach=16/47 regex=0 abbrevs=['StPO']
    unreachable: ['1B_192/2022 E. 4.1.2', '1B_195/2022 E. 2.2.1', '1B_211/2017 E. 2.1', '1B_572/2021 E. 2.1', '1B_581/2022 E. 2.1.2', '1B_88/2022 E. 2.1']
  val[3] oracle=0.095 k=11 reach=2/10 regex=0 abbrevs=[]
    unreachable: ['Art. 100 Abs. 1 BGG', 'Art. 458 Abs. 3 ZGB', 'Art. 467 ZGB', 'Art. 469 Abs. 1 ZGB', 'Art. 469 Abs. 2 ZGB', 'Art. 505 Abs. 1 ZGB']
  val[4] oracle=0.154 k=2 reach=1/11 regex=0 abbrevs=[]
    unreachable: ['Art. 133 Abs. 1 ZGB', 'Art. 133 Abs. 2 ZGB', 'Art. 273 Abs. 1 ZGB', 'Art. 274 Abs. 2 ZGB', 'Art. 285 Abs. 1 ZGB', 'BGE 126 III 

100%|██████████| 40/40 [00:04<00:00,  9.43it/s]


  Val F1 = 0.1775  |  Rows = 40
    query_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                predicted_citations
0   test_001                                                                                                              Art. 963 Abs. 1 ZGB;Art. 8 ZGB;Art. 655 Abs. 2 ZGB;Art. 965 Abs. 3 ZGB;Art. 10 Abs. 2 URG;Art. 527 ZGB;Art. 965 Abs. 2 ZGB;Art. 2 Abs. 2 URG;Art. 1 UWG;Art. 241 Abs. 3 ZGB;Art. 969 Abs. 1 ZGB;Art. 946 Abs. 1 ZGB;Art. 2 Ab